In [1]:
import os
import pandas as pd
import shutil
from glob import glob
from sklearn.model_selection import train_test_split

# --- 1. SETUP KAGGLE E DOWNLOAD ---
# RICORDATI DI INSERIRE LE TUE CREDENZIALI QUI SOTTO:
os.environ['KAGGLE_USERNAME'] = "tommasocentonze"
os.environ['KAGGLE_KEY'] = "KGAT_6c5f85ead94e49c5c6cb47ab4a2a31c9"

print("Download del dataset cellercity/vindr-pcxr-zipped...")
!kaggle datasets download -d cellercity/vindr-pcxr-zipped

print("Estrazione in corso (potrebbe volerci un po')...")
os.makedirs('vindr_data', exist_ok=True)
!unzip -q vindr-pcxr-zipped.zip -d vindr_data
print("Estrazione completata!")


# --- 2. LETTURA CSV E FILTRAGGIO CLASSI ---
BASE_PATH = './vindr_data'
DATASET_NAME = "VinDr_Target_DA_Ready"

# Trova il file CSV delle etichette globali che hai esplorato prima
csv_files = glob(os.path.join(BASE_PATH, '**', 'image_labels_train.csv'), recursive=True)
if not csv_files:
    raise FileNotFoundError("File image_labels_train.csv non trovato! Assicurati che l'estrazione sia andata a buon fine.")
CSV_PATH = csv_files[0]

print("Filtraggio casi PNEUMONIA e NORMAL in corso...")
df = pd.read_csv(CSV_PATH)

# Uniamo i 3 tipi di polmonite e i casi sani (basato sui dati del tuo CSV)
malati_mask = (df['Pneumonia'] == 1.0) | (df['Brocho-pneumonia'] == 1.0) | (df['Pleuro-pneumonia'] == 1.0)
sani_mask = (df['No finding'] == 1.0)

df_malati = df[malati_mask].copy()
df_malati['target_label'] = 'PNEUMONIA'

df_sani = df[sani_mask].copy()
df_sani['target_label'] = 'NORMAL'

# Uniamo in un unico DataFrame pulito
df_clean = pd.concat([df_malati, df_sani])
print(f"Totale immagini selezionate: {len(df_clean)} ({len(df_malati)} PNEUMONIA, {len(df_sani)} NORMAL)")


# --- 3. SPLIT TRAIN (80%), VAL (10%), TEST (10%) ---
# Usiamo stratify per mantenere le proporzioni (sbilanciate, ma gestite poi dal tuo WeightedRandomSampler)
train_df, temp_df = train_test_split(df_clean, test_size=0.2, stratify=df_clean['target_label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['target_label'], random_state=42)

splits = {'train': train_df, 'val': val_df, 'test': test_df}


# --- 4. CREAZIONE CARTELLE E COPIA IMMAGINI ---
print("Mappatura delle immagini fisiche PNG...")
# Cerchiamo tutte le immagini png nelle sottocartelle
all_images = glob(os.path.join(BASE_PATH, '**', '*.png'), recursive=True)

# Creiamo un dizionario "nome_immagine_senza_estensione" -> "percorso_completo"
img_dict = {os.path.splitext(os.path.basename(img))[0]: img for img in all_images}
print(f"Trovate {len(img_dict)} immagini PNG nel dataset estratto.")

for split_name, split_df in splits.items():
    print(f"Creazione split '{split_name}'...")
    os.makedirs(os.path.join(DATASET_NAME, split_name, 'NORMAL'), exist_ok=True)
    os.makedirs(os.path.join(DATASET_NAME, split_name, 'PNEUMONIA'), exist_ok=True)

    copiate = 0
    perse = 0
    for _, row in split_df.iterrows():
        img_id = row['image_id']
        label_folder = row['target_label']

        # Copiamo l'immagine se esiste nel dizionario
        if img_id in img_dict:
            src_path = img_dict[img_id]
            dest_path = os.path.join(DATASET_NAME, split_name, label_folder, os.path.basename(src_path))
            shutil.copy(src_path, dest_path)
            copiate += 1
        else:
            perse += 1

    print(f"  -> {copiate} immagini copiate, {perse} non trovate.")


# --- 5. COMPRESSIONE E PULIZIA ---
print("Compressione del nuovo dataset in corso...")
shutil.make_archive(DATASET_NAME, 'zip', DATASET_NAME)

# Pulizia per liberare spazio su Colab
shutil.rmtree(DATASET_NAME)
shutil.rmtree('vindr_data')
os.remove('vindr-pcxr-zipped.zip')

print(f"\nFATTO! ✅ Scarica il file '{DATASET_NAME}.zip' dalla colonna a sinistra di Colab.")

Download del dataset cellercity/vindr-pcxr-zipped...
Dataset URL: https://www.kaggle.com/datasets/cellercity/vindr-pcxr-zipped
License(s): unknown
100% 31.6G/31.6G [18:24<00:00, 30.7MB/s]

Estrazione in corso (potrebbe volerci un po')...
Estrazione completata!
Filtraggio casi PNEUMONIA e NORMAL in corso...
Totale immagini selezionate: 6063 (920 PNEUMONIA, 5143 NORMAL)
Mappatura delle immagini fisiche PNG...
Trovate 9123 immagini PNG nel dataset estratto.
Creazione split 'train'...
  -> 4848 immagini copiate, 2 non trovate.
Creazione split 'val'...
  -> 606 immagini copiate, 0 non trovate.
Creazione split 'test'...
  -> 607 immagini copiate, 0 non trovate.
Compressione del nuovo dataset in corso...

FATTO! ✅ Scarica il file 'VinDr_Target_DA_Ready.zip' dalla colonna a sinistra di Colab.
